# Customer Churn Prediction — Week 2: Feature Engineering & Model Building

**Goal:** Clean and encode features, handle class imbalance, train 3 models, compare performance, and explain predictions with SHAP.

**Input:** `data/telco_churn_merged.csv` (saved in Week 1)  
**Target:** `Churn_binary` — 1 = Churned, 0 = Retained


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)
from sklearn.utils.class_weight import compute_class_weight
import shap
import joblib

print('All libraries loaded!')

## 2. Load the merged dataset

In [ ]:
df = pd.read_csv('../data/telco_churn_merged.csv')
print(f'Dataset shape: {df.shape}')
df.head(3)

## 3. Select and clean features

In [ ]:
# Drop columns not useful for modeling
# - IDs, geographic details, free-text reasons, and leakage columns
drop_cols = [
    'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
    'Lat Long', 'Latitude', 'Longitude',
    'Churn Label', 'Churn Value', 'Churn Score', 'Churn Reason', 'Churn Category',
    'CLTV'  # business metric calculated after churn — data leakage
]

# Only drop columns that exist
drop_cols = [c for c in drop_cols if c in df.columns]
df_model = df.drop(columns=drop_cols).copy()

print(f'Modeling dataset shape: {df_model.shape}')
print(f'\nColumns kept: {list(df_model.columns)}')

## 4. Feature engineering

In [ ]:
# Create new features from existing ones

# 1. Tenure group — bin tenure into categories
df_model['Tenure Group'] = pd.cut(
    df_model['Tenure Months'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months']
)

# 2. Avg monthly revenue per GB (internet efficiency)
if 'Avg Monthly GB Download' in df_model.columns:
    df_model['Revenue per GB'] = (
        df_model['Monthly Charges'] /
        (df_model['Avg Monthly GB Download'].replace(0, np.nan))
    ).fillna(0)

# 3. Has multiple services (bundled customer flag)
service_cols = ['Online Security', 'Online Backup', 'Device Protection',
                'Tech Support', 'Streaming TV', 'Streaming Movies']
service_cols = [c for c in service_cols if c in df_model.columns]
df_model['Num Services'] = df_model[service_cols].apply(
    lambda row: sum(row == 'Yes'), axis=1
)

print('New features created:')
print('  - Tenure Group')
print('  - Revenue per GB')
print('  - Num Services')
df_model[['Tenure Months', 'Tenure Group', 'Num Services']].head()

## 5. Encode categorical variables

In [ ]:
# Binary encode Yes/No columns
binary_cols = [
    'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
    'Multiple Lines', 'Online Security', 'Online Backup',
    'Device Protection', 'Tech Support', 'Streaming TV',
    'Streaming Movies', 'Paperless Billing', 'Married',
    'Referred a Friend' if 'Referred a Friend' in df_model.columns else None
]
binary_cols = [c for c in binary_cols if c and c in df_model.columns]

for col in binary_cols:
    df_model[col] = df_model[col].map({'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0})

# Label encode remaining categorical columns
cat_cols = df_model.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'Churn_binary']

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = df_model[col].astype(str)
    df_model[col] = le.fit_transform(df_model[col])

print(f'Categorical columns encoded: {cat_cols}')
print(f'\nFinal dataset shape: {df_model.shape}')
print(f'Missing values: {df_model.isnull().sum().sum()}')

## 6. Prepare features and target

In [ ]:
X = df_model.drop(columns=['Churn_binary'])
y = df_model['Churn_binary']

# Fill any remaining NaN
X = X.fillna(0)

# Train/test split — stratified to preserve churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]:,} rows')
print(f'Test set     : {X_test.shape[0]:,} rows')
print(f'Churn rate in train: {y_train.mean():.1%}')
print(f'Churn rate in test : {y_test.mean():.1%}')

## 7. Handle class imbalance

In [ ]:
# Compute class weights to handle 73.5% vs 26.5% imbalance
classes = np.array([0, 1])
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = {0: weights[0], 1: weights[1]}

print(f'Class weights: {class_weight_dict}')
print('Retained (0) weight:', round(weights[0], 3))
print('Churned  (1) weight:', round(weights[1], 3))
print('\nThis tells the model to pay more attention to churned customers.')

## 8. Train 3 models

In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Model 1: Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
print('Logistic Regression trained!')

# Model 2: Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print('Random Forest trained!')

# Model 3: XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb.fit(X_train, y_train)
print('XGBoost trained!')

## 9. Evaluate all models

In [ ]:
def evaluate_model(name, model, X_test, y_test, scaled=False):
    X = X_test_scaled if scaled else X_test
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    
    return {
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1 Score':  round(f1_score(y_test, y_pred), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, y_prob), 4)
    }

results = [
    evaluate_model('Logistic Regression', lr, X_test_scaled, y_test, scaled=True),
    evaluate_model('Random Forest',       rf, X_test, y_test),
    evaluate_model('XGBoost',             xgb, X_test, y_test),
]

results_df = pd.DataFrame(results).set_index('Model')
print('\n===== Model Comparison =====')
print(results_df.to_string())

In [ ]:
# Plot model comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#3498DB', '#2ECC71', '#E74C3C']

for i, (_, row) in enumerate(results_df.iterrows()):
    ax.bar(x + i * width, [row[m] for m in metrics],
           width, label=row.name, color=colors[i], alpha=0.85)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')

## 10. Confusion matrix — best model

In [ ]:
# Use XGBoost as best model (update if results show otherwise)
best_model = xgb
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'], ax=ax)
ax.set_title('XGBoost — Confusion Matrix')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Retained', 'Churned']))

## 11. ROC Curve comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_estimator(lr,  X_test_scaled, y_test, ax=ax, name='Logistic Regression')
RocCurveDisplay.from_estimator(rf,  X_test,        y_test, ax=ax, name='Random Forest')
RocCurveDisplay.from_estimator(xgb, X_test,        y_test, ax=ax, name='XGBoost')

ax.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax.set_title('ROC Curve Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('../data/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. SHAP — explain what drives churn

In [ ]:
# SHAP values for XGBoost
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

# Summary plot — top features driving churn
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False,
                  max_display=15)
plt.title('Top 15 Features Driving Churn (SHAP)')
plt.tight_layout()
plt.savefig('../data/shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP chart saved!')

In [ ]:
# SHAP dot plot — shows direction of impact
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, show=False, max_display=15)
plt.title('SHAP Feature Impact — Direction and Magnitude')
plt.tight_layout()
plt.savefig('../data/shap_dot_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Save the best model

In [ ]:
# Save model and scaler for use in the Streamlit app
joblib.dump(xgb,    '../data/best_model.pkl')
joblib.dump(scaler, '../data/scaler.pkl')
joblib.dump(list(X.columns), '../data/feature_names.pkl')

print('Saved:')
print('  data/best_model.pkl')
print('  data/scaler.pkl')
print('  data/feature_names.pkl')
print(f'\nFeatures used: {len(X.columns)}')

## 14. Results summary

Fill this in after running all cells:

**Best model:** XGBoost (or update based on your results)  

**Performance (test set):**

| Model | Accuracy | F1 Score | AUC-ROC |
|---|---|---|---|
| Logistic Regression | — | — | — |
| Random Forest | — | — | — |
| XGBoost | — | — | — |

**Top 3 churn drivers (from SHAP):**
> Write the top 3 features from the SHAP bar chart here.

**Key insight:**
> In one sentence — what is the most actionable finding from this model?
> e.g. "Customers on month-to-month contracts with low satisfaction scores
> and short tenure are 3x more likely to churn."